In [1]:
!pip install nilearn openneuro-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 80.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.6.1 which is incompatible.
bigframes 1.36.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.


In [2]:
!openneuro-py download --dataset ds003643 --include "stimuli/task-lppCN_section_*" --target-dir "/kaggle/working/ds003643-download/stimuli"  --verify-hash


👋 Hello! This is openneuro-py 2024.2.0. Great to see you! 🤗

   👉 Please report problems 🤯 and bugs 🪲 at
      https://github.com/hoechenberger/openneuro-py/issues

🌍 Preparing to download ds003643 …
📁 Traversing directories for ds003643 : 6302 entities [02:12, 47.58 entities/s] 
📥 Retrieving up to 14 files (5 concurrent downloads). 
participants.json:   0%|                              | 0.00/427 [00:00<?, ?B/s]
                                                                                
task-lppCN_section_1.wav:   2%|▏            | 785k/47.5M [00:00<00:06, 7.52MB/s]
task-lppCN_section_4.wav:   0%|                     | 0.00/51.5M [00:00<?, ?B/s]

task-lppCN_section_2.wav:   0%|                     | 0.00/54.1M [00:00<?, ?B/s]


task-lppCN_section_1.wav:  10%|█▏          | 4.95M/47.5M [00:00<00:01, 28.2MB/s]



task-lppCN_section_5.wav:   0%|                     | 0.00/49.2M [00:00<?, ?B/s]
task-lppCN_section_4.wav:   1%|             | 271k/51.5M [00:00<00:20, 2.65MB/s]

task-lpp

In [5]:
import os
from tqdm import tqdm
import torchaudio
from transformers import WhisperFeatureExtractor, WhisperModel
import numpy as np
import shutil
import torch.nn as nn
import torch


import math
import nibabel as nib
import nilearn
from nilearn import plotting, image
import matplotlib.pyplot as plt
import pandas as pd

In [6]:
# Set constants
STIMULI_DIR = "/kaggle/working/ds003643-download/stimuli/stimuli/"
SECTIONS = [f"task-lppCN_section_{i}.wav" for i in range(1, 10)]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [7]:
# Load Whisper model and feature extractor once
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-medium")
model = WhisperModel.from_pretrained("openai/whisper-medium").to(device)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

In [8]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Load processor and model
processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium").to(device)

# Force English decoding (if audio is in English)
language = "english"
task = "transcribe"

decoded_texts = []

# Loop through each section
for section in tqdm(SECTIONS):
    file_path = os.path.join(STIMULI_DIR, section)

    # Load and preprocess audio
    speech_array, sampling_rate = torchaudio.load(file_path)
    speech_array = torchaudio.functional.resample(speech_array, sampling_rate, 16000)

    input_features = processor(speech_array.squeeze(), sampling_rate=16000, return_tensors="pt").input_features.to(device)

    # Generate token ids
    predicted_ids = model.generate(input_features)

    # Decode to text
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    decoded_texts.append((section, transcription))
    print(f"{section}: {transcription}")


tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

...

task-lppCN_section_7.wav: 在沙漠、岩石、雪地上行走了很长的时间以后,小王子终于发现了一条大路,所有的大路都是通往人住的地方的。


 89%|████████▉ | 8/9 [00:29<00:03,  3.31s/it]

task-lppCN_section_8.wav: 你好小王子说你好商人说的这是一位贩卖能够止渴的精致药丸的商人每周通服一丸就不会感觉口渴你为什么卖这玩意儿小王子说这就大大的节约了时间商人说专家们计算过这样每周可以节约53分钟那么用这53分钟做什么用随便怎么用都行


100%|██████████| 9/9 [00:32<00:00,  3.64s/it]

task-lppCN_section_9.wav: 在井旁邊有一堵殘缺的石牆第二天晚上我工作回來的時候我遠遠地看見了小王子拉拉著雙腿坐在牆上我聽見他在說話你怎麼不記得了呢絕不是在這大概還有另一個聲音在回答他因為他打著槍說道沒錯沒錯日子是對的但地點不是這裡我繼續朝牆走去我還是看不到
